# 2 — Preprocessing
**Raw data:** `data/diabetes.csv` — read-only, never modified  
**Output:** `data/preprocessed_data/`  
**Split:** 90 / 10 stratified on `Outcome`

All imputation and scaling statistics are fit on **train only** to prevent data leakage.

## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

RAW_PATH   = Path('../data/diabetes.csv')
OUT_DIR    = Path('../data/preprocessed_data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE    = 0.10

# columns where 0 encodes a missing value
ZERO_AS_NAN = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# columns that get log1p (skew > 1.5 in EDA)
LOG_COLS = ['Insulin', 'DiabetesPedigreeFunction']

df_raw = pd.read_csv(RAW_PATH)
print('Raw shape:', df_raw.shape)
df_raw.head()

## 2. Zero → NaN
Five columns use `0` as a placeholder for missing values where zero is biologically impossible.
`Pregnancies` (valid zero) and `Age` (min 21 in this dataset) are left untouched.

In [ ]:
df = df_raw.copy()
df[ZERO_AS_NAN] = df[ZERO_AS_NAN].replace(0, np.nan)

print('Missing values after zero replacement:')
print(df.isnull().sum())

## 3. Train / Test Split  (90 / 10, stratified)
Split **before** any imputation or scaling to ensure no leakage from test into train statistics.

In [ ]:
X = df.drop(columns='Outcome')
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test:  {X_test.shape[0]} rows  ({X_test.shape[0]/len(X)*100:.1f}%)')
print()
print('Class balance — train:')
print((y_train.value_counts(normalize=True)*100).round(1))
print('Class balance — test:')
print((y_test.value_counts(normalize=True)*100).round(1))

## 4. Missing Value Imputation

| Column | Strategy | Reason (from EDA) |
|---|---|---|
| Insulin | Median per class | 48.7% missing; class medians differ by ~67 units |
| SkinThickness | Median per class | 29.6% missing; class medians differ |
| BloodPressure | Overall median | 4.6% missing; class medians differ only 4.5 units |
| BMI | Overall median | 1.4% missing; straightforward fill |
| Glucose | Overall median | Only 5 missing; strongest predictor — conservative fill |

All medians computed from **train set only**.

In [ ]:
# columns that use per-class median imputation
CLASS_MEDIAN_COLS   = ['Insulin', 'SkinThickness']
# columns that use overall median imputation
OVERALL_MEDIAN_COLS = ['BloodPressure', 'BMI', 'Glucose']

# ── compute statistics on train only ─────────────────────────────────────────
class_medians   = {}
overall_medians = {}

train_with_target = X_train.copy()
train_with_target['Outcome'] = y_train.values

for col in CLASS_MEDIAN_COLS:
    class_medians[col] = train_with_target.groupby('Outcome')[col].median()

for col in OVERALL_MEDIAN_COLS:
    overall_medians[col] = X_train[col].median()

print('Class medians (train):')
print(pd.DataFrame(class_medians))
print()
print('Overall medians (train):')
print(pd.Series(overall_medians).round(2))

In [ ]:
def impute(X, y_labels, class_medians, overall_medians):
    X = X.copy()
    # per-class median
    for col, medians in class_medians.items():
        for cls, val in medians.items():
            mask = X[col].isna() & (y_labels.values == cls)
            X.loc[mask, col] = val
    # overall median
    for col, val in overall_medians.items():
        X[col] = X[col].fillna(val)
    return X

X_train_imp = impute(X_train, y_train, class_medians, overall_medians)
X_test_imp  = impute(X_test,  y_test,  class_medians, overall_medians)

print('Missing after imputation — train:', X_train_imp.isnull().sum().sum())
print('Missing after imputation — test: ', X_test_imp.isnull().sum().sum())

## 5. Log1p Transform
`Insulin` (skew 2.17) and `DiabetesPedigreeFunction` (skew 1.92) are right-skewed.  
`log1p` compresses the long tail and reduces leverage of high-value outliers.

In [ ]:
X_train_imp[LOG_COLS] = np.log1p(X_train_imp[LOG_COLS])
X_test_imp[LOG_COLS]  = np.log1p(X_test_imp[LOG_COLS])

print('Skewness after log1p (train):')
print(X_train_imp[LOG_COLS].skew().round(3))

## 6. StandardScaler
Fit on **train** only; transform both splits.  
Saved to disk for use in downstream model notebooks.

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imp),
    columns=X_train_imp.columns,
    index=X_train_imp.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_imp),
    columns=X_test_imp.columns,
    index=X_test_imp.index
)

print('Scaled train — mean (should be ~0):')
print(X_train_scaled.mean().round(3))
print()
print('Scaled train — std  (should be ~1):')
print(X_train_scaled.std().round(3))

## 7. Save to `data/preprocessed_data/`

In [ ]:
# unscaled (for tree-based models)
X_train_imp.to_csv(OUT_DIR / 'X_train.csv',  index=True)
X_test_imp.to_csv( OUT_DIR / 'X_test.csv',   index=True)
y_train.to_csv(    OUT_DIR / 'y_train.csv',  index=True, header=True)
y_test.to_csv(     OUT_DIR / 'y_test.csv',   index=True, header=True)

# scaled (for linear models / SVM / KNN)
X_train_scaled.to_csv(OUT_DIR / 'X_train_scaled.csv', index=True)
X_test_scaled.to_csv( OUT_DIR / 'X_test_scaled.csv',  index=True)

# scaler artifact
joblib.dump(scaler, OUT_DIR / 'scaler.pkl')

print('Saved to', OUT_DIR.resolve())
print('Files:', [f.name for f in sorted(OUT_DIR.iterdir())])

## 8. Sanity Check

In [ ]:
summary = pd.DataFrame({
    'split':   ['train', 'test'],
    'n_rows':  [X_train_imp.shape[0], X_test_imp.shape[0]],
    'pct':     [f'{X_train_imp.shape[0]/len(X)*100:.1f}%',
                f'{X_test_imp.shape[0]/len(X)*100:.1f}%'],
    'pos_rate':[f'{y_train.mean()*100:.1f}%', f'{y_test.mean()*100:.1f}%'],
    'any_nan': [X_train_imp.isnull().any().any(), X_test_imp.isnull().any().any()]
})
print(summary.to_string(index=False))
print()
print('Raw file untouched — shape:', pd.read_csv(RAW_PATH).shape)